# Clase 7 — Herencia en POO

**Objetivo:** Entender qué es la herencia, para qué sirve, y cómo se implementa en Python.

---

## 1. Repaso rápido: ¿qué vimos la clase pasada?

La semana pasada creamos nuestra primera **clase** en Python: `Carta`.

Una **clase** es un molde. Un **objeto** es algo creado con ese molde.

```python
class Carta:
    def __init__(self):
        self.nombre = ""
        self.fuerza = 0
        self.coste = 0
        self.habilidad = ""
```

Cada `Carta` que creábamos con `c = Carta()` era un **objeto independiente** con sus propios atributos.

In [ ]:
# Repaso: creamos una carta manualmente
class Carta:
    def __init__(self, nombre, fuerza, coste):
        self.nombre = nombre
        self.fuerza = fuerza
        self.coste = coste

    def __str__(self):
        return f"{self.nombre} (fuerza: {self.fuerza}, coste: {self.coste})"

    def atacar(self, otra):
        print(f"{self.nombre} ataca a {otra.nombre}!")

c1 = Carta("Link", 4, 2)
c2 = Carta("Samus", 3, 3)

print(c1)
print(c2)
c1.atacar(c2)

---

## 2. El problema: muchas clases similares

Imagina que queremos tener **tipos** de cartas distintos:

- Carta normal
- **Carta Legendaria** (tiene rareza y brilla)
- **Carta Jefe** (tiene mucha vida y puede contraatacar)

Opción A — Copiar y pegar la clase `Carta` tres veces:

```python
class CartaNormal:    # nombre, fuerza, coste, atacar()...
class CartaLegendaria: # nombre, fuerza, coste, rareza, atacar()... (REPETIDO)
class CartaJefe:      # nombre, fuerza, coste, vida, atacar()... (REPETIDO)
```

Esto es un problema. Si mañana quiero cambiar `atacar()`, tengo que cambiarlo en **tres lugares**.

> **Regla de oro en programación:** No te repitas. *(Don't Repeat Yourself — DRY)*

Opción B — **Herencia**: una clase base con lo común, y clases hijas con lo particular.

---

## 3. ¿Qué es la Herencia?

La **herencia** permite que una clase **hija** tome todo lo que ya tiene una clase **padre**, y le agregue o modifique cosas.

```
        Carta          ← clase PADRE (base)
       /     \
  CartaLeg  CartaJefe  ← clases HIJAS
```

Las clases hijas **heredan** automáticamente todos los atributos y métodos del padre.

### Sintaxis en Python:

```python
class ClaseHija(ClasePadre):
    ...
```

Solo eso. Los paréntesis indican de quién hereda.

In [ ]:
# Ejemplo mínimo: CartaLegendaria hereda de Carta

class CartaLegendaria(Carta):
    pass  # por ahora no agrega nada

# Aunque no definimos nada, CartaLegendaria tiene TODO lo de Carta
leg = CartaLegendaria("Master Chief", 8, 5)
print(leg)           # usa el __str__ de Carta
leg.atacar(c1)       # usa el atacar() de Carta
print(type(leg))     # CartaLegendaria
print(isinstance(leg, Carta))  # True — sigue siendo una Carta

> `isinstance(objeto, Clase)` devuelve `True` si el objeto **es** de esa clase o de alguna hija de ella.

---

## 4. Extendiendo el `__init__`: agregar atributos propios

Para que la clase hija tenga sus **propios atributos adicionales**, necesitamos escribir su propio `__init__`.

Pero también queremos conservar lo del padre. Para eso usamos **`super()`**.

```python
class CartaLegendaria(Carta):
    def __init__(self, nombre, fuerza, coste, rareza):
        super().__init__(nombre, fuerza, coste)  # llama al __init__ del padre
        self.rareza = rareza                      # agrega lo propio
```

`super()` = "el padre de esta clase". Evita repetir código del padre.

In [ ]:
class CartaLegendaria(Carta):
    def __init__(self, nombre, fuerza, coste, rareza):
        super().__init__(nombre, fuerza, coste)  # hereda atributos del padre
        self.rareza = rareza  # nuevo atributo propio

    def __str__(self):
        return f"[LEGENDARIA ★{self.rareza}] {self.nombre} (fuerza: {self.fuerza})"

leg = CartaLegendaria("Master Chief", 8, 5, 3)
print(leg)
leg.atacar(c2)  # hereda atacar() del padre sin problema

print("\n--- Comparación ---")
print("Es CartaLegendaria?", isinstance(leg, CartaLegendaria))
print("Es Carta?",           isinstance(leg, Carta))

---

## 5. Sobreescritura de métodos *(Method Overriding)*

Una clase hija puede **reemplazar** (sobreescribir) un método del padre con su propia versión.

Python siempre usa el método **más específico** disponible:
- Si la hija tiene el método → usa el de la hija
- Si no → sube al padre

### Ejemplo: `CartaJefe` con un `atacar()` diferente

In [ ]:
class CartaJefe(Carta):
    def __init__(self, nombre, fuerza, coste, vida):
        super().__init__(nombre, fuerza, coste)
        self.vida = vida

    def __str__(self):
        return f"[JEFE] {self.nombre} (fuerza: {self.fuerza}, vida: {self.vida})"

    def atacar(self, otra):
        # sobreescribe el atacar del padre con comportamiento especial
        print(f"{self.nombre} APLASTA a {otra.nombre} con fuerza {self.fuerza * 2}!")
        otra.fuerza -= 1  # además debilita al rival

jefe = CartaJefe("Bowser", 9, 6, 30)
objetivo = Carta("Toad", 2, 1)

print(jefe)
print(f"Fuerza de {objetivo.nombre} antes: {objetivo.fuerza}")
jefe.atacar(objetivo)
print(f"Fuerza de {objetivo.nombre} después: {objetivo.fuerza}")

---

## 6. Extender un método del padre con `super()`

¿Y si queremos que la hija haga **lo mismo que el padre más algo extra**? Usamos `super()` dentro del método también.

In [ ]:
class CartaVeneno(Carta):
    def __init__(self, nombre, fuerza, coste):
        super().__init__(nombre, fuerza, coste)
        self.veneno = 2  # daño de veneno por turno

    def atacar(self, otra):
        super().atacar(otra)             # primero hace el ataque normal del padre
        print(f"... y aplica {self.veneno} de veneno!")  # luego agrega lo suyo
        otra.fuerza -= self.veneno

veneno = CartaVeneno("Joker", 5, 3)
victima = Carta("Mario", 4, 2)

print(f"Fuerza de {victima.nombre} antes: {victima.fuerza}")
veneno.atacar(victima)
print(f"Fuerza de {victima.nombre} después: {victima.fuerza}")

---

## 7. Resumen visual

```
Carta
├── __init__(nombre, fuerza, coste)
├── __str__()
└── atacar(otra)
         │
         ├── CartaLegendaria(Carta)
         │       ├── __init__ → super() + self.rareza
         │       └── __str__  → sobreescribe
         │
         ├── CartaJefe(Carta)
         │       ├── __init__ → super() + self.vida
         │       ├── __str__  → sobreescribe
         │       └── atacar() → sobreescribe completamente
         │
         └── CartaVeneno(Carta)
                 ├── __init__ → super() + self.veneno
                 └── atacar() → super().atacar() + comportamiento extra
```

| Concepto | Qué hace |
|---|---|
| `class Hija(Padre)` | La hija hereda todo del padre |
| `super().__init__(...)` | Llama al constructor del padre |
| Redefinir un método | Sobreescribe (reemplaza) el del padre |
| `super().metodo()` | Llama al método del padre desde la hija |
| `isinstance(obj, Clase)` | Verifica si un objeto es de esa clase (o hija) |

---

## 8. Ejercicio: agregar un tipo de carta nuevo

Crea una clase `CartaCuradora` que herede de `Carta` y:

1. Tenga un atributo extra `curacion` (un número entero)
2. Tenga un método `curar(otra)` que suba la fuerza de otra carta en `self.curacion`
3. Sobreescriba `__str__` para mostrar que es una carta curadora

Pruébala creando un objeto y llamando a `curar()` sobre otra carta.

In [ ]:
# Tu código aquí

class CartaCuradora(Carta):
    pass  # reemplaza esto